### Assignment 1 (3 scores):

- Use the Numpy library only to construct the Linear Regression model.
- Train and Evaluate that Linear Regression model on the [Forest Fires](https://archive.ics.uci.edu/static/public/162/forest+fires.zip) dataset.

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("forest+fires/forestfires.csv")
df.head()

,X,Y,month,day,FFMC,DMC,DC,ISI,temp,RH,wind,rain,area
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2,0.0
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0,0.0


In [3]:
df_ohe = pd.get_dummies(df, columns=['month', 'day'], dtype=float)
feature_cols = [c for c in df_ohe.columns if c != 'area']
scale_cols = ['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']
scale_idx = [feature_cols.index(c) for c in scale_cols]

X_raw = df_ohe[feature_cols].to_numpy(dtype=float)
y = df_ohe['area'].to_numpy(dtype=float)

np.random.seed(10)
indices = np.random.permutation(len(X_raw))
split = int(0.8 * len(X_raw))
train_idx, test_idx = indices[:split], indices[split:]

X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()

mean = np.mean(X_train[:, scale_idx], axis=0)
std = np.std(X_train[:, scale_idx], axis=0)
X_train[:, scale_idx] = (X_train[:, scale_idx] - mean) / std
X_test[:, scale_idx] = (X_test[:, scale_idx] - mean) / std

y_train_log = np.log1p(y_train)
print(X_train.shape, X_test.shape)

(413, 29) (104, 29)


In [4]:
class LinearRegressionNP:
    def __init__(self):
        self.w = None

    def fit(self, X, y):
        X_b = np.c_[np.ones((len(X), 1)), X]
        self.w = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y

    def predict(self, X):
        X_b = np.c_[np.ones((len(X), 1)), X]
        return X_b @ self.w

def evaluate(y_true, y_pred):
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)
    return mse, rmse, mae, r2

model_a1 = LinearRegressionNP()
model_a1.fit(X_train, y_train_log)
y_pred_log = model_a1.predict(X_test)
y_pred_a1 = np.expm1(y_pred_log)

mse_a1, rmse_a1, mae_a1, r2_a1 = evaluate(y_test, y_pred_a1)
print(f"MSE: {mse_a1:.4f}")
print(f"RMSE: {rmse_a1:.4f}")
print(f"MAE: {mae_a1:.4f}")
print(f"R2: {r2_a1:.4f}")

MSE: 773.0546
RMSE: 27.8039
MAE: 10.1716
R2: -0.0690


### Assignment 2 (3 scores):

- Standardize the data so that their mean is $0$ and their variance is $1$.
- Compare the results of Linear Regression model when being trained on the original and standardized data.

In [5]:
mean_all = np.mean(X_train_raw, axis=0)
std_all = np.std(X_train_raw, axis=0)

X_train_std = (X_train_raw - mean_all) / std_all
X_test_std = (X_test_raw - mean_all) / std_all

print("Mean (first 10):", np.round(np.mean(X_train_std, axis=0)[:10], 4))
print("Variance (first 10):", np.round(np.var(X_train_std, axis=0)[:10], 4))

Mean (first 10): [ 0. -0.  0. -0. -0.  0. -0.  0. -0.  0.]
Variance (first 10): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [6]:
model_orig = LinearRegressionNP()
model_orig.fit(X_train_raw, y_train_log)
y_pred_orig = np.expm1(model_orig.predict(X_test_raw))
mse_orig, rmse_orig, mae_orig, r2_orig = evaluate(y_test, y_pred_orig)

model_std = LinearRegressionNP()
model_std.fit(X_train_std, y_train_log)
y_pred_std = np.expm1(model_std.predict(X_test_std))
mse_std, rmse_std, mae_std, r2_std = evaluate(y_test, y_pred_std)

compare_a2 = pd.DataFrame({
    'Model': ['Original Data', 'Standardized Data (All Features)', 'Assignment 1 (Z-score 8 Features)'],
    'MSE': [mse_orig, mse_std, mse_a1],
    'RMSE': [rmse_orig, rmse_std, rmse_a1],
    'MAE': [mae_orig, mae_std, mae_a1],
    'R2': [r2_orig, r2_std, r2_a1]
})
compare_a2

,Model,MSE,RMSE,MAE,R2
0,Original Data,773.051044,27.803795,10.171694,-0.069009
1,Standardized Data (All Features),773.130364,27.805222,10.174241,-0.069119
2,Assignment 1 (Z-score 8 Features),773.054555,27.803859,10.171573,-0.069014


**Comparison:**
- The error metrics (MSE, RMSE, MAE, R2) are almost identical across the original and standardized datasets because the Normal Equation (OLS) is invariant to linear scaling of features.
- Standardizing features ensures mean $0$ and variance $1$, puts all weights on a comparable scale, and improves numerical stability.

### Assignment 3 (2 scores):

- Perform feature engineering to remove columns that are linearly dependent (colinear) with other columns in the dataset.
- Evaluate the Linear Regression model when beeing trained on the original data and preprocessed data.

In [7]:
df_prep = pd.get_dummies(df, columns=['month', 'day'], drop_first=True, dtype=float).drop(columns=['DC'])

prep_cols = [c for c in df_prep.columns if c != 'area']
prep_idx = [prep_cols.index(c) for c in scale_cols if c in prep_cols]

X_prep = df_prep[prep_cols].to_numpy(dtype=float)
X_prep_train, X_prep_test = X_prep[train_idx].copy(), X_prep[test_idx].copy()

m_p = np.mean(X_prep_train[:, prep_idx], axis=0)
s_p = np.std(X_prep_train[:, prep_idx], axis=0)
X_prep_train[:, prep_idx] = (X_prep_train[:, prep_idx] - m_p) / s_p
X_prep_test[:, prep_idx] = (X_prep_test[:, prep_idx] - m_p) / s_p

model_prep = LinearRegressionNP()
model_prep.fit(X_prep_train, y_train_log)
y_pred_prep = np.expm1(model_prep.predict(X_prep_test))

mse_prep, rmse_prep, mae_prep, r2_prep = evaluate(y_test, y_pred_prep)

pd.DataFrame({
    'Dữ liệu': ['Dữ liệu gốc (OHE đầy đủ)', 'Dữ liệu đã xử lý (bỏ cột phụ thuộc tuyến tính & DC)'],
    'MSE': [mse_a1, mse_prep],
    'RMSE': [rmse_a1, rmse_prep],
    'MAE': [mae_a1, mae_prep],
    'R2': [r2_a1, r2_prep]
})

,Dữ liệu,MSE,RMSE,MAE,R2
0,Dữ liệu gốc (OHE đầy đủ),773.054555,27.803859,10.171573,-0.069014
1,Dữ liệu đã xử lý (bỏ cột phụ thuộc tuyến tính ...,775.328966,27.844730,10.068079,-0.072159


**Nhận xét:**
- Dùng `drop_first=True` khi One-hot Encoding để bỏ các cột giả bị phụ thuộc tuyến tính hoàn toàn (bẫy biến giả) và loại bỏ thêm cột `DC` do có tương quan cao với `DMC`.
- Sau khi xử lý, mô hình ổn định hơn về mặt đại số (ma trận đủ hạng) và chỉ số `MAE` giảm nhẹ từ `10.1716` xuống `10.0681`.

### Assignment 4 (2 scores):

- Implement the Linear Regression model using Machine Learning libraries (Scikit Learn or Skorch).
- Compare the results of Linear Regression model constructed manually and from Machine Learning libraries.

In [8]:
from sklearn.linear_model import LinearRegression

sk_model = LinearRegression()
sk_model.fit(X_train, y_train_log)
y_pred_sk = np.expm1(sk_model.predict(X_test))

mse_sk, rmse_sk, mae_sk, r2_sk = evaluate(y_test, y_pred_sk)
print("Độ lệch dự đoán lớn nhất:", np.max(np.abs(y_pred_a1 - y_pred_sk)))

pd.DataFrame({
    'Mô hình': ['Tự cài đặt bằng Numpy', 'Thư viện Scikit-Learn'],
    'MSE': [mse_a1, mse_sk],
    'RMSE': [rmse_a1, rmse_sk],
    'MAE': [mae_a1, mae_sk],
    'R2': [r2_a1, r2_sk]
})

Độ lệch dự đoán lớn nhất: 0.0012209164260825744


,Mô hình,MSE,RMSE,MAE,R2
0,Tự cài đặt bằng Numpy,773.054555,27.803859,10.171573,-0.069014
1,Thư viện Scikit-Learn,773.051044,27.803795,10.171694,-0.069009


**Nhận xét:**
- Mô hình `LinearRegression` của Scikit-Learn cho kết quả các chỉ số (`MSE`, `RMSE`, `MAE`, `R2`) gần như trùng khớp hoàn toàn với mô hình tự cài đặt bằng Numpy.
- Độ lệch dự đoán lớn nhất giữa hai mô hình chỉ khoảng `0.0012` do sai số làm tròn số thực và cách giải khác nhau (SVD trong Scikit-Learn so với giả nghịch đảo `pinv` trong Numpy).